In [14]:
import chess
from typing import Optional, List

class SearchNode:
    """Tree node for search visualization"""
    def __init__(self, board: chess.Board, move: Optional[chess.Move] = None, 
                 eval_score: int = 0, depth: int = 0):
        self.board = board.copy()
        self.move = move
        self.eval_score = eval_score
        self.depth = depth
        self.children: List['SearchNode'] = []
    
    def add_child(self, child: 'SearchNode'):
        self.children.append(child)


class ChessPosition:
    """Wrapper for python-chess Board"""
    
    def __init__(self, board: Optional[chess.Board] = None):
        self.board = board if board else chess.Board()
    
    def evaluate(self) -> int:
        """Material count evaluation"""
        material = 0
        piece_values = {
            chess.PAWN: 100,
            chess.KNIGHT: 320,
            chess.BISHOP: 330,
            chess.ROOK: 500,
            chess.QUEEN: 900,
            chess.KING: 10000
        }
        
        for square in chess.SQUARES:
            piece = self.board.piece_at(square)
            if piece:
                value = piece_values[piece.piece_type]
                material += value if piece.color == chess.WHITE else -value
        
        # Return from perspective of current player
        return material if self.board.turn == chess.WHITE else -material
    
    def get_forcing_moves(self) -> List[chess.Move]:
        """Get only captures"""
        return [move for move in self.board.legal_moves 
                if self.board.is_capture(move)]
    
    def make_move(self, move: chess.Move) -> None:
        self.board.push(move)
    
    def undo_move(self) -> None:
        self.board.pop()
    
    def display(self):
        print(self.board)
        print(f"Material: {self.evaluate():+3d}")



In [15]:
def quiescent_with_tree(position: ChessPosition, alpha: int, beta: int, 
                        depth: int = 0, max_depth: int = 2, 
                        parent_node: Optional[SearchNode] = None) -> tuple:
    """Quiescent search building a tree"""
    
    stand_pat = position.evaluate()
    is_maximizing = position.board.turn == chess.WHITE
    
    # Create node for this position
    current_node = SearchNode(
        board=position.board,
        move=None,
        eval_score=stand_pat,
        depth=depth
    )
    
    if parent_node is not None:
        parent_node.add_child(current_node)
    
    # Standing pat checks
    if is_maximizing:
        if stand_pat >= beta:
            return beta, current_node
        alpha = max(alpha, stand_pat)
    else:
        if stand_pat <= alpha:
            return alpha, current_node
        beta = min(beta, stand_pat)
    
    if depth >= max_depth:
        return stand_pat, current_node
    
    forcing_moves = position.get_forcing_moves()
    
    if not forcing_moves:
        return stand_pat, current_node
    
    # Sort by MVV-LVA
    piece_values = {
        chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
        chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 100
    }
    
    def mvv_lva_score(move):
        victim = position.board.piece_at(move.to_square)
        attacker = position.board.piece_at(move.from_square)
        victim_val = piece_values.get(victim.piece_type, 0) if victim else 0
        attacker_val = piece_values.get(attacker.piece_type, 0) if attacker else 0
        return 100 * victim_val - attacker_val
    
    forcing_moves.sort(key=mvv_lva_score, reverse=True)
    
    best_eval = stand_pat
    
    for move in forcing_moves:
        position.make_move(move)
        
        eval_score, child_node = quiescent_with_tree(
            position,
            alpha,
            beta,
            depth + 1,
            max_depth,
            current_node
        )
        
        child_node.move = move
        child_node.eval_score = eval_score
        
        position.undo_move()
        
        if is_maximizing:
            best_eval = max(best_eval, eval_score)
            alpha = max(alpha, best_eval)
            if alpha >= beta:
                break
        else:
            best_eval = min(best_eval, eval_score)
            beta = min(beta, best_eval)
            if beta <= alpha:
                break
    
    current_node.eval_score = best_eval
    return best_eval, current_node

In [23]:
def draw_tree_chess(node: SearchNode, max_children: int = 10):
    """Draw tree vertically with full board display"""
    
    def render_node(board: chess.Board, move: Optional[chess.Move], 
                   eval_score: int, board_size: int = 8) -> List[str]:
        """Render single node with board"""
        lines = []
        
        if move:
            move_str = move.uci()
        else:
            move_str = "START"
        
        # Header
        lines.append(f"┌{'─' * 12}┐")
        lines.append(f"│ {move_str:8s} [{eval_score:+4d}] │")
        lines.append(f"├{'─' * 12}┤")
        
        # Board with unicode pieces
        board_lines = board.unicode(invert_color=False, borders=True).split('\n')
        for board_line in board_lines:
            lines.append(f"│ {board_line} │")
        
        lines.append(f"└{'─' * 12}┘")
        
        return lines
    
    def get_tree_width(node: SearchNode, max_children_arg: int) -> int:
        """Calculate width needed for this subtree"""
        sorted_children = sorted(
            node.children,
            key=lambda n: n.eval_score,
            reverse=True
        )[:max_children_arg]
        
        if not sorted_children:
            return 16  # Width of a node box
        
        child_widths = [get_tree_width(child, max_children_arg) for child in sorted_children]
        return max(16, sum(child_widths) + (len(sorted_children) - 1) * 2)
    
    def render_tree(node: SearchNode, max_children_arg: int = 3, 
                   depth: int = 0) -> List[str]:
        """Recursively render tree vertically"""
        
        output = []
        
        # Current node
        node_lines = render_node(node.board, node.move, node.eval_score)
        output.extend(node_lines)
        
        # Get top children by eval
        sorted_children = sorted(
            node.children,
            key=lambda n: n.eval_score,
            reverse=True
        )[:max_children_arg]
        
        if not sorted_children:
            return output
        
        # Add vertical connector
        output.append("      │      ")
        
        # Render all children
        child_renders = []
        for child in sorted_children:
            child_render = render_tree(child, max_children_arg, depth + 1)
            child_renders.append(child_render)
        
        # Calculate positioning
        child_widths = [max(len(line) for line in child_render) for child_render in child_renders]
        total_width = sum(child_widths) + (len(sorted_children) - 1) * 2
        
        # Draw horizontal connector line
        if len(sorted_children) > 1:
            first_offset = child_widths[0] // 2
            line = " " * first_offset + "┬"
            for i in range(1, len(sorted_children)):
                prev_width = child_widths[i - 1]
                gap = 2
                line += "─" * (prev_width - first_offset - 1) + "┬"
                first_offset = 0
            line += "─" * (child_widths[-1] // 2 - 1)
            output.append(line)
        else:
            output.append("      │      ")
        
        # Add children with proper spacing
        max_child_height = max(len(r) for r in child_renders)
        for line_idx in range(max_child_height):
            line = ""
            for child_idx, (child_render, child_width) in enumerate(zip(child_renders, child_widths)):
                if line_idx < len(child_render):
                    child_line = child_render[line_idx]
                else:
                    child_line = ""
                
                # Add connector for first line
                if line_idx == 0 and len(sorted_children) > 1:
                    line += "│" + " " * (child_width - 2) + "│"
                elif line_idx == 0:
                    line += "│" + " " * (child_width - 2) + "│"
                else:
                    line += child_line.ljust(child_width)
                
                if child_idx < len(sorted_children) - 1:
                    line += "  "
            
            output.append(line.rstrip())
        
        return output
    
    tree_lines = render_tree(node, max_children_arg=max_children)
    for line in tree_lines:
        print(line)

In [26]:
# Run it:
if __name__ == "__main__":
    # Create position from real game
    pos = ChessPosition()
    pos.board.push_san("e4")
    pos.board.push_san("e5")
    pos.board.push_san("Nf3")
    pos.board.push_san("Nc6")
    
    print("=" * 70)
    print("POSITION: 1.e4 e5 2.Nf3 Nc6")
    print("=" * 70)
    pos.display()
    
    print("\n" + "=" * 70)
    print("QUIESCENT SEARCH TREE (full depth=3)")
    print("=" * 70 + "\n")
    
    result, tree_root = quiescent_with_tree(
        pos,
        alpha=-10000,
        beta=10000,
        max_depth=8
    )
    
    print(f"Best evaluation: {result:+8d}\n")
    
    draw_tree_chess(tree_root, max_children=20)  # Show all children

POSITION: 1.e4 e5 2.Nf3 Nc6
r . b q k b n r
p p p p . p p p
. . n . . . . .
. . . . p . . .
. . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q K B . R
Material:  +0

QUIESCENT SEARCH TREE (full depth=3)

Best evaluation:       +0

┌────────────┐
│ START    [  +0] │
├────────────┤
│   ----------------- │
│ 8 |♜|⭘|♝|♛|♚|♝|♞|♜| │
│   ----------------- │
│ 7 |♟|♟|♟|♟|⭘|♟|♟|♟| │
│   ----------------- │
│ 6 |⭘|⭘|♞|⭘|⭘|⭘|⭘|⭘| │
│   ----------------- │
│ 5 |⭘|⭘|⭘|⭘|♟|⭘|⭘|⭘| │
│   ----------------- │
│ 4 |⭘|⭘|⭘|⭘|♙|⭘|⭘|⭘| │
│   ----------------- │
│ 3 |⭘|⭘|⭘|⭘|⭘|♘|⭘|⭘| │
│   ----------------- │
│ 2 |♙|♙|♙|♙|⭘|♙|♙|♙| │
│   ----------------- │
│ 1 |♖|♘|♗|♕|♔|♗|⭘|♖| │
│   ----------------- │
│    a b c d e f g h │
└────────────┘
      │      
      │      
│                     │
│ f3e5     [  +0] │
├────────────┤
│   ----------------- │
│ 8 |♜|⭘|♝|♛|♚|♝|♞|♜| │
│   ----------------- │
│ 7 |♟|♟|♟|♟|⭘|♟|♟|♟| │
│   ----------------- │
│ 6 |⭘|⭘|♞|⭘|⭘|⭘|⭘|⭘| │
│   ----------------- │
│ 5 |⭘|⭘|⭘|⭘|

In [27]:
def quiescent_with_tree_no_prune(position: ChessPosition, alpha: int, beta: int, 
                        depth: int = 0, max_depth: int = 2, 
                        parent_node: Optional[SearchNode] = None) -> tuple:
    """Quiescent search building a tree - NO PRUNING"""
    
    stand_pat = position.evaluate()
    is_maximizing = position.board.turn == chess.WHITE
    
    current_node = SearchNode(
        board=position.board,
        move=None,
        eval_score=stand_pat,
        depth=depth
    )
    
    if parent_node is not None:
        parent_node.add_child(current_node)
    
    if depth >= max_depth:
        return stand_pat, current_node
    
    forcing_moves = position.get_forcing_moves()
    
    if not forcing_moves:
        return stand_pat, current_node
    
    # Sort by MVV-LVA
    piece_values = {
        chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
        chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 100
    }
    
    def mvv_lva_score(move):
        victim = position.board.piece_at(move.to_square)
        attacker = position.board.piece_at(move.from_square)
        victim_val = piece_values.get(victim.piece_type, 0) if victim else 0
        attacker_val = piece_values.get(attacker.piece_type, 0) if attacker else 0
        return 100 * victim_val - attacker_val
    
    forcing_moves.sort(key=mvv_lva_score, reverse=True)
    
    best_eval = stand_pat
    
    # SEARCH ALL MOVES - NO PRUNING
    for move in forcing_moves:
        position.make_move(move)
        
        eval_score, child_node = quiescent_with_tree_no_prune(
            position,
            alpha,
            beta,
            depth + 1,
            max_depth,
            current_node
        )
        
        child_node.move = move
        child_node.eval_score = eval_score
        
        position.undo_move()
        
        # Update best eval but DON'T PRUNE
        if is_maximizing:
            best_eval = max(best_eval, eval_score)
        else:
            best_eval = min(best_eval, eval_score)
    
    current_node.eval_score = best_eval
    return best_eval, current_node

In [28]:
result, tree_root = quiescent_with_tree_no_prune(
    pos,
    alpha=-10000,
    beta=10000,
    max_depth=8
)

print(f"Best evaluation: {result:+8d}\n")
draw_tree_chess(tree_root, max_children=20)

Best evaluation:  +0

┌────────────┐
│ START    [  +0] │
├────────────┤
│   ----------------- │
│ 8 |♜|⭘|♝|♛|♚|♝|♞|♜| │
│   ----------------- │
│ 7 |♟|♟|♟|♟|⭘|♟|♟|♟| │
│   ----------------- │
│ 6 |⭘|⭘|♞|⭘|⭘|⭘|⭘|⭘| │
│   ----------------- │
│ 5 |⭘|⭘|⭘|⭘|♟|⭘|⭘|⭘| │
│   ----------------- │
│ 4 |⭘|⭘|⭘|⭘|♙|⭘|⭘|⭘| │
│   ----------------- │
│ 3 |⭘|⭘|⭘|⭘|⭘|♘|⭘|⭘| │
│   ----------------- │
│ 2 |♙|♙|♙|♙|⭘|♙|♙|♙| │
│   ----------------- │
│ 1 |♖|♘|♗|♕|♔|♗|⭘|♖| │
│   ----------------- │
│    a b c d e f g h │
└────────────┘
      │      
      │      
│                     │
│ f3e5     [-220] │
├────────────┤
│   ----------------- │
│ 8 |♜|⭘|♝|♛|♚|♝|♞|♜| │
│   ----------------- │
│ 7 |♟|♟|♟|♟|⭘|♟|♟|♟| │
│   ----------------- │
│ 6 |⭘|⭘|♞|⭘|⭘|⭘|⭘|⭘| │
│   ----------------- │
│ 5 |⭘|⭘|⭘|⭘|♘|⭘|⭘|⭘| │
│   ----------------- │
│ 4 |⭘|⭘|⭘|⭘|♙|⭘|⭘|⭘| │
│   ----------------- │
│ 3 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘| │
│   ----------------- │
│ 2 |♙|♙|♙|♙|⭘|♙|♙|♙| │
│   ----------------- │
│ 1 |♖|♘|♗|♕|♔|♗|⭘|♖| │
│   -------